# Resolve_AGENT: восстановление связи `(user_id, course_id) -> users_course_id`

Цель ноутбука: восстановить отсутствующий в `users_courses.csv` технический ключ `users_course_id`, чтобы корректно связывать `users_courses` с `user_lessons`, `wk_users_courses_actions` и `user_access_histories`.

Ограничения: `EDA.ipynb`, `Resolve_id.ipynb`, `data/raw`, `Hipotises.md` и `README.md` не изменяются; результаты сохраняются только в `data/AGENT`; если связь не восстановлена однозначно, остается `NA`.


## Что найдено в существующих ноутбуках и план

`EDA.ipynb`: CSV имеют технический индекс `Unnamed: 0`; id приводятся через `pd.to_numeric(...).astype("Int64")`; даты приводятся через `pd.to_datetime`; учителя исключаются по `users.type`, где `User::Agent` считается teacher/account-agent.

`Resolve_id.ipynb`: уже проверялась идея восстановления через пересечение `course_id` пользователей одной `(lesson_id, group_id)` в `user_lessons`. Эта идея полезна, но одна не покрывает все пары.

План: загрузить только нужные колонки, повторить очистку id/дат/teacher filter, построить независимые сигналы `points`, `lesson_count`, `access_date`, `group_intersection`, `user_singleton`, оставить только однозначные связи, удалить конфликты, посчитать покрытие и сохранить bridge в `data/AGENT`.


## Предположения и правила однозначности

Сигналы:

* `points`: `users_courses.wk_points` совпадает с суммой `user_lessons.wk_points` по `users_course_id`;
* `lesson_count`: `users_courses.wk_max_viewable_lessons` совпадает с числом уникальных `lesson_id` по `users_course_id`;
* `access_date`: `users_courses.access_finished_at` совпадает с `min` или `max` `user_access_histories.access_expired_at` по `users_course_id`;
* `group_intersection`: курс входит в пересечение `course_id` всех пользователей одной `(lesson_id, group_id)`;
* `user_singleton`: у пользователя ровно один `course_id` и ровно один наблюдаемый `users_course_id`.

Правила: принимаем только однозначные связи; после объединения сигналов удаляем любые конфликты; `user_access_histories` не содержит `user_id`, поэтому до восстановления bridge ее нельзя полностью очистить от учителей.


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 240)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
AGENT_DIR = PROJECT_ROOT / "data" / "AGENT"
AGENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"DATA_DIR={DATA_DIR}")
print(f"AGENT_DIR={AGENT_DIR}")


PROJECT_ROOT=C:\Repos\Xakaton
DATA_DIR=C:\Repos\Xakaton\data\raw
AGENT_DIR=C:\Repos\Xakaton\data\AGENT


## 1. Аудит схемы и загрузка нужных колонок

Сначала смотрим заголовки CSV без загрузки больших таблиц целиком. Для восстановления bridge используются только таблицы, где есть хотя бы одна сторона связи или teacher filter: `users_courses`, `user_lessons`, `wk_users_courses_actions`, `user_access_histories`, `users`.


In [2]:
# Smotrim tolko shemu, bez zagruzki bolshih tablic celikom.
schema_rows = []
for csv_path in sorted(DATA_DIR.glob("*.csv")):
    header = pd.read_csv(csv_path, nrows=0, encoding="utf-8").columns.tolist()
    schema_rows.append({"table": csv_path.stem, "column_count": len(header), "columns": header})

schema_df = pd.DataFrame(schema_rows)
display(schema_df)

used_tables = {"users_courses", "user_lessons", "wk_users_courses_actions", "user_access_histories", "users"}
excluded_schema_notes = pd.DataFrame([
    {"table": row["table"], "reason": "used" if row["table"] in used_tables else "not used: no direct users_course_id/course bridge for this task"}
    for _, row in schema_df.iterrows()
])
display(excluded_schema_notes)


,table,column_count,columns
0,award_badges,7,"[Unnamed: 0, name, title, level, quota, special, unlocked_small_image_url]"
1,lessons,12,"[Unnamed: 0, course_id, conspect_expected, task_expected, lesson_number, wk_max_points, wk_task_count, wk_survival_training_expected, wk_scratch_playground_enabled, wk_attendance_tracking_enabled, wk_video_duration, wk_attendance_tracki..."
2,user_access_histories,5,"[Unnamed: 0, users_course_id, access_started_at, access_expired_at, activator_class]"
3,user_activity_histories,4,"[Unnamed: 0, user_lesson_id, action, created_at]"
4,user_answers,14,"[Unnamed: 0, user_id, task_id, attempts, solved, points, max_attempts, results, skipped, resource_type, submitted_at, wk_partial_answer, performance, async_check_status]"
5,user_award_badges,4,"[Unnamed: 0, award_badge_id, user_id, created_at]"
6,user_lessons,12,"[Unnamed: 0, user_id, lesson_id, group_id, video_visited, translation_visited, users_course_id, solved, solved_tasks_count, wk_points, video_viewed, wk_solved_task_count]"
7,user_trainings,13,"[Unnamed: 0, user_id, training_id, solved_tasks_count, earned_points, type, state, submitted_answers_count, started_at, finished_at, attempts, mark, mark_saved_at]"
8,users,22,"[Unnamed: 0, id, last_explainer_seen_→_course, created_at, updated_at, type, remember_created_at, sign_in_count, current_sign_in_at, last_sign_in_at, grade_id, subscribed, grade_checked, is_teacher, timezone, grade_changed_at, xp, d_wk_..."
9,users_courses,13,"[Unnamed: 0, user_id, course_id, state, created_at, updated_at, access_finished_at, wk_points, wk_max_points, wk_max_viewable_lessons, wk_max_task_count, wk_officially_started_at, wk_course_completed_at]"


,table,reason
0,award_badges,not used: no direct users_course_id/course bridge for this task
1,lessons,not used: no direct users_course_id/course bridge for this task
2,user_access_histories,used
3,user_activity_histories,not used: no direct users_course_id/course bridge for this task
4,user_answers,not used: no direct users_course_id/course bridge for this task
5,user_award_badges,not used: no direct users_course_id/course bridge for this task
6,user_lessons,used
7,user_trainings,not used: no direct users_course_id/course bridge for this task
8,users,used
9,users_courses,used


In [3]:
TABLE_SPECS = {
    "users_courses": {"filename": "users_courses.csv", "usecols": ["user_id", "course_id", "wk_points", "wk_max_viewable_lessons", "access_finished_at"]},
    "user_lessons": {"filename": "user_lessons.csv", "usecols": ["user_id", "lesson_id", "group_id", "users_course_id", "wk_points"]},
    "wk_users_courses_actions": {"filename": "wk_users_courses_actions.csv", "usecols": ["user_id", "users_course_id"]},
    "user_access_histories": {"filename": "user_access_histories.csv", "usecols": ["users_course_id", "access_expired_at"]},
    "users": {"filename": "users.csv", "usecols": ["id", "type"]},
}

dfs = {}
load_rows = []
for table_name, spec in TABLE_SPECS.items():
    df = pd.read_csv(DATA_DIR / spec["filename"], usecols=spec["usecols"], encoding="utf-8", low_memory=False)
    dfs[table_name] = df
    load_rows.append({"table": table_name, "rows": len(df), "cols": df.shape[1]})

load_summary = pd.DataFrame(load_rows).sort_values("table").reset_index(drop=True)
display(load_summary)


,table,rows,cols
0,user_access_histories,667124,2
1,user_lessons,3070664,5
2,users,95395,2
3,users_courses,290835,5
4,wk_users_courses_actions,12909207,2


## 2. Техническая очистка: id, даты, teacher filter

Повторяем нужную часть EDA: числовые id со строковыми запятыми приводятся к `Int64`, даты нормализуются до дня, teacher/account-agent удаляются из таблиц с `user_id`.


In [4]:
# Perevodim id-stolbcy v Int64, kak v EDA.
def to_int_id(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
    else:
        numeric = pd.to_numeric(series.astype("string").str.replace(",", "", regex=False), errors="coerce")
    return numeric.astype("Int64")

INT_COLS = {
    "users_courses": ["user_id", "course_id"],
    "user_lessons": ["user_id", "lesson_id", "group_id", "users_course_id"],
    "wk_users_courses_actions": ["user_id", "users_course_id"],
    "user_access_histories": ["users_course_id"],
    "users": ["id"],
}

for table_name, cols in INT_COLS.items():
    for col in cols:
        dfs[table_name][col] = to_int_id(dfs[table_name][col])

for table_name, cols in {"users_courses": ["wk_points", "wk_max_viewable_lessons"], "user_lessons": ["wk_points"]}.items():
    for col in cols:
        dfs[table_name][col] = pd.to_numeric(dfs[table_name][col], errors="coerce")

# Dati normalizuem do dnya, tak kak access_expired_at v odnoj tablice bez vremeni.
dfs["users_courses"]["access_finished_date"] = pd.to_datetime(dfs["users_courses"]["access_finished_at"], errors="coerce").dt.normalize()
dfs["user_access_histories"]["access_expired_date"] = pd.to_datetime(dfs["user_access_histories"]["access_expired_at"], errors="coerce").dt.normalize()

raw_users_courses_pairs = dfs["users_courses"][["user_id", "course_id"]].dropna().drop_duplicates().shape[0]

# Teacher filter po users.type == Agent-like, kak v EDA.
dfs["users"]["user_id"] = dfs["users"]["id"].copy()
teacher_mask = dfs["users"]["type"].astype("string").str.contains("Agent", na=False)
teacher_ids = set(dfs["users"].loc[teacher_mask, "user_id"].dropna().astype("int64"))

teacher_filter_rows = []
for table_name in ["users_courses", "user_lessons", "wk_users_courses_actions"]:
    before = len(dfs[table_name])
    dfs[table_name] = dfs[table_name].loc[~dfs[table_name]["user_id"].isin(teacher_ids)].copy()
    after = len(dfs[table_name])
    teacher_filter_rows.append({"table": table_name, "rows_before": before, "rows_after": after, "rows_removed": before - after})

users_before = len(dfs["users"])
dfs["users"] = dfs["users"].loc[~dfs["users"]["user_id"].isin(teacher_ids)].copy()
teacher_filter_rows.append({"table": "users", "rows_before": users_before, "rows_after": len(dfs["users"]), "rows_removed": users_before - len(dfs["users"])})

teacher_filter_stats = pd.DataFrame(teacher_filter_rows)
print(f"teacher_ids={len(teacher_ids)}")
display(teacher_filter_stats)


teacher_ids=4748


,table,rows_before,rows_after,rows_removed
0,users_courses,290835,267206,23629
1,user_lessons,3070664,3008595,62069
2,wk_users_courses_actions,12909207,12633684,275523
3,users,95395,90647,4748


## 3. Базовые множества id и проверки консистентности

Проверяем, что в activity-таблицах каждый `users_course_id` принадлежит ровно одному `user_id`. Это обязательная предпосылка для bridge.


In [5]:
uc_pairs = (
    dfs["users_courses"][["user_id", "course_id", "wk_points", "wk_max_viewable_lessons", "access_finished_date"]]
    .dropna(subset=["user_id", "course_id"])
    .drop_duplicates(subset=["user_id", "course_id"])
    .reset_index(drop=True)
)

ul_base = (
    dfs["user_lessons"][["user_id", "lesson_id", "group_id", "users_course_id", "wk_points"]]
    .dropna(subset=["user_id", "users_course_id"])
    .drop_duplicates()
    .reset_index(drop=True)
)

wka_pairs = (
    dfs["wk_users_courses_actions"][["user_id", "users_course_id"]]
    .dropna(subset=["user_id", "users_course_id"])
    .drop_duplicates()
    .reset_index(drop=True)
)

access_base = (
    dfs["user_access_histories"][["users_course_id", "access_expired_date"]]
    .dropna(subset=["users_course_id"])
    .drop_duplicates()
    .reset_index(drop=True)
)

all_user_ucid = pd.concat([ul_base[["user_id", "users_course_id"]], wka_pairs[["user_id", "users_course_id"]]], ignore_index=True).drop_duplicates().reset_index(drop=True)

id_integrity = pd.DataFrame([
    {"table": "user_lessons", "unique_users_course_id": ul_base["users_course_id"].nunique(), "max_user_id_per_users_course_id": ul_base.groupby("users_course_id")["user_id"].nunique().max()},
    {"table": "wk_users_courses_actions", "unique_users_course_id": wka_pairs["users_course_id"].nunique(), "max_user_id_per_users_course_id": wka_pairs.groupby("users_course_id")["user_id"].nunique().max()},
    {"table": "combined_activity_ids", "unique_users_course_id": all_user_ucid["users_course_id"].nunique(), "max_user_id_per_users_course_id": all_user_ucid.groupby("users_course_id")["user_id"].nunique().max()},
])

base_summary = pd.DataFrame([
    {"metric": "raw_users_courses_pairs_before_teacher_filter", "value": raw_users_courses_pairs},
    {"metric": "student_users_courses_pairs_after_teacher_filter", "value": len(uc_pairs)},
    {"metric": "user_lessons_rows_for_mapping", "value": len(ul_base)},
    {"metric": "wk_actions_user_ucid_pairs", "value": len(wka_pairs)},
    {"metric": "observed_student_user_ucid_pairs", "value": len(all_user_ucid)},
])

display(base_summary)
display(id_integrity)


,metric,value
0,raw_users_courses_pairs_before_teacher_filter,290835
1,student_users_courses_pairs_after_teacher_filter,267206
2,user_lessons_rows_for_mapping,3008595
3,wk_actions_user_ucid_pairs,208565
4,observed_student_user_ucid_pairs,208685


,table,unique_users_course_id,max_user_id_per_users_course_id
0,user_lessons,208672,1
1,wk_users_courses_actions,208565,1
2,combined_activity_ids,208685,1


## 4. Сигналы `points`, `lesson_count`, `access_date`

Строим компактную таблицу кандидатов внутри каждого пользователя: все наблюдаемые `users_course_id` пользователя умножаются на все его `course_id` из `users_courses`. Затем проверяются совпадения баллов, числа уроков и даты окончания доступа.


In [6]:
# Aggregaty user_lessons na urovne (user_id, users_course_id).
ul_agg = (
    ul_base
    .groupby(["user_id", "users_course_id"], dropna=False)
    .agg(ul_rows=("lesson_id", "size"), ul_lesson_nunique=("lesson_id", "nunique"), ul_points_sum=("wk_points", "sum"))
    .reset_index()
)

access_agg = (
    access_base
    .groupby("users_course_id")
    .agg(access_expired_min=("access_expired_date", "min"), access_expired_max=("access_expired_date", "max"), access_rows=("users_course_id", "size"))
    .reset_index()
)

candidate_cross = (
    all_user_ucid
    .merge(uc_pairs[["user_id", "course_id", "wk_points", "wk_max_viewable_lessons", "access_finished_date"]], on="user_id", how="inner")
    .merge(ul_agg, on=["user_id", "users_course_id"], how="left")
    .merge(access_agg, on="users_course_id", how="left")
)

candidate_cross["points_match"] = ((candidate_cross["wk_points"].fillna(0) - candidate_cross["ul_points_sum"].fillna(0)).abs().le(1e-6) & candidate_cross["ul_rows"].notna())
candidate_cross["lesson_count_match"] = (candidate_cross["wk_max_viewable_lessons"].notna() & candidate_cross["ul_lesson_nunique"].notna() & candidate_cross["wk_max_viewable_lessons"].eq(candidate_cross["ul_lesson_nunique"]))
candidate_cross["access_date_match"] = (candidate_cross["access_finished_date"].notna() & (candidate_cross["access_finished_date"].eq(candidate_cross["access_expired_min"]) | candidate_cross["access_finished_date"].eq(candidate_cross["access_expired_max"])))

signal_rows = []
for signal_name in ["points_match", "lesson_count_match", "access_date_match"]:
    signal_edges = candidate_cross.loc[candidate_cross[signal_name], ["user_id", "course_id", "users_course_id"]].drop_duplicates()
    signal_rows.append({
        "signal": signal_name,
        "candidate_edges": len(signal_edges),
        "candidate_user_ucid": signal_edges[["user_id", "users_course_id"]].drop_duplicates().shape[0],
        "candidate_user_course": signal_edges[["user_id", "course_id"]].drop_duplicates().shape[0],
    })

signal_summary = pd.DataFrame(signal_rows)
print(f"candidate_cross_edges={len(candidate_cross)}")
display(signal_summary)


candidate_cross_edges=777191


,signal,candidate_edges,candidate_user_ucid,candidate_user_course
0,points_match,212767,202970,209216
1,lesson_count_match,15833,15353,15731
2,access_date_match,315885,208294,232029


## 5. Сигнал `group_intersection`

Структурированная версия идеи из `Resolve_id.ipynb`: для каждой `(lesson_id, group_id)` берем пользователей группы, смотрим их `course_id` из `users_courses`, оставляем курсы, которые есть у всех участников группы, затем переносим этот сигнал на `(user_id, users_course_id)`.


In [7]:
# Gotovim unikalnye stroki dlya gruppovogo signala.
ul_group_base = ul_base[["user_id", "lesson_id", "group_id", "users_course_id"]].dropna(subset=["user_id", "lesson_id", "group_id", "users_course_id"]).drop_duplicates().reset_index(drop=True)

group_members = ul_group_base[["lesson_id", "group_id", "user_id"]].drop_duplicates()
group_sizes = group_members.groupby(["lesson_id", "group_id"])["user_id"].nunique().rename("group_size").reset_index()

group_member_courses = group_members.merge(uc_pairs[["user_id", "course_id"]], on="user_id", how="left").dropna(subset=["course_id"])

group_course_support = (
    group_member_courses
    .groupby(["lesson_id", "group_id", "course_id"])["user_id"]
    .nunique()
    .rename("course_support")
    .reset_index()
    .merge(group_sizes, on=["lesson_id", "group_id"], how="left")
)

group_course_intersection = group_course_support.loc[group_course_support["course_support"].eq(group_course_support["group_size"]), ["lesson_id", "group_id", "course_id"]].copy()

ul_group_candidates = ul_group_base.merge(group_course_intersection, on=["lesson_id", "group_id"], how="left").dropna(subset=["course_id"])
user_ucid_group_count = ul_group_base.groupby(["user_id", "users_course_id"]).size().rename("pair_group_count").reset_index()

user_ucid_course_support = (
    ul_group_candidates
    .drop_duplicates(["user_id", "users_course_id", "lesson_id", "group_id", "course_id"])
    .groupby(["user_id", "users_course_id", "course_id"])
    .size()
    .rename("support")
    .reset_index()
    .merge(user_ucid_group_count, on=["user_id", "users_course_id"], how="left")
)

group_edges = user_ucid_course_support.loc[user_ucid_course_support["support"].eq(user_ucid_course_support["pair_group_count"]), ["user_id", "course_id", "users_course_id"]].drop_duplicates()

group_signal_summary = pd.DataFrame([
    {"metric": "lesson_group_count", "value": group_sizes.shape[0]},
    {"metric": "lesson_groups_with_course_intersection", "value": group_course_intersection[["lesson_id", "group_id"]].drop_duplicates().shape[0]},
    {"metric": "group_candidate_edges", "value": len(group_edges)},
    {"metric": "group_candidate_user_ucid", "value": group_edges[["user_id", "users_course_id"]].drop_duplicates().shape[0]},
])
display(group_signal_summary)


,metric,value
0,lesson_group_count,7478
1,lesson_groups_with_course_intersection,7461
2,group_candidate_edges,200107
3,group_candidate_user_ucid,164459


## 6. Объединение evidence и строгая проверка однозначности

Применяются два слоя защиты: `force_unique` оставляет только one-to-one связи внутри конкретного сигнала, а `remove_conflicts` после объединения сигналов удаляет любые спорные связи.


In [8]:
# Funkciya ostavlyaet tolko lokalno odnoznachnye rebra.
def force_unique(edges: pd.DataFrame) -> pd.DataFrame:
    keep_cols = ["user_id", "course_id", "users_course_id"]
    edges = edges[keep_cols].drop_duplicates().copy()
    if edges.empty:
        return edges

    courses_per_ucid = edges.groupby(["user_id", "users_course_id"])["course_id"].nunique().rename("course_candidates").reset_index()
    ucids_per_course = edges.groupby(["user_id", "course_id"])["users_course_id"].nunique().rename("ucid_candidates").reset_index()
    unique_edges = edges.merge(courses_per_ucid, on=["user_id", "users_course_id"], how="left").merge(ucids_per_course, on=["user_id", "course_id"], how="left")

    return unique_edges.loc[unique_edges["course_candidates"].eq(1) & unique_edges["ucid_candidates"].eq(1), keep_cols].drop_duplicates()

# Funkciya ubiraet konfliktuyuschie svyazi posle union signalov.
def remove_conflicts(edges: pd.DataFrame):
    edges = edges.drop_duplicates().copy()
    conflict_user_course = edges.groupby(["user_id", "course_id"])["users_course_id"].nunique().rename("users_course_id_count").reset_index().query("users_course_id_count > 1")
    conflict_user_ucid = edges.groupby(["user_id", "users_course_id"])["course_id"].nunique().rename("course_id_count").reset_index().query("course_id_count > 1")
    clean = edges.merge(conflict_user_course[["user_id", "course_id"]].assign(_conflict_user_course=1), on=["user_id", "course_id"], how="left").merge(conflict_user_ucid[["user_id", "users_course_id"]].assign(_conflict_user_ucid=1), on=["user_id", "users_course_id"], how="left")
    clean = clean.loc[clean["_conflict_user_course"].isna() & clean["_conflict_user_ucid"].isna()].drop(columns=["_conflict_user_course", "_conflict_user_ucid"])
    return clean, conflict_user_course, conflict_user_ucid


evidence_parts = []
for signal_name, mask_col in [("points", "points_match"), ("lesson_count", "lesson_count_match"), ("access_date", "access_date_match")]:
    signal_edges = candidate_cross.loc[candidate_cross[mask_col], ["user_id", "course_id", "users_course_id"]].drop_duplicates().copy()
    signal_edges["evidence"] = signal_name
    evidence_parts.append(signal_edges)

group_evidence = group_edges.copy()
group_evidence["evidence"] = "group_intersection"
evidence_parts.append(group_evidence)

evidence_long = pd.concat(evidence_parts, ignore_index=True).drop_duplicates()
edge_evidence = evidence_long.groupby(["user_id", "course_id", "users_course_id"])["evidence"].agg(lambda s: "|".join(sorted(set(s)))).reset_index()
edge_evidence["evidence_count"] = edge_evidence["evidence"].str.count("\\|") + 1

method_edges = {
    "points_unique": force_unique(edge_evidence.loc[edge_evidence["evidence"].str.contains("points", regex=False)]),
    "date_unique": force_unique(edge_evidence.loc[edge_evidence["evidence"].str.contains("access_date", regex=False)]),
    "group_unique": force_unique(edge_evidence.loc[edge_evidence["evidence"].str.contains("group_intersection", regex=False)]),
    "evidence2_unique": force_unique(edge_evidence.loc[edge_evidence["evidence_count"].ge(2)]),
}

user_course_count = uc_pairs.groupby("user_id")["course_id"].nunique().rename("course_count").reset_index()
user_ucid_count = all_user_ucid.groupby("user_id")["users_course_id"].nunique().rename("observed_users_course_id_count").reset_index()

user_singleton_edges = (
    user_course_count
    .merge(user_ucid_count, on="user_id", how="inner")
    .query("course_count == 1 and observed_users_course_id_count == 1")[["user_id"]]
    .merge(uc_pairs[["user_id", "course_id"]], on="user_id", how="inner")
    .merge(all_user_ucid, on="user_id", how="inner")[["user_id", "course_id", "users_course_id"]]
    .drop_duplicates()
)
method_edges["user_singleton"] = user_singleton_edges

method_summary = pd.DataFrame([{"method": name, "resolved_edges_before_global_conflict_check": len(edges)} for name, edges in method_edges.items()]).sort_values("resolved_edges_before_global_conflict_check", ascending=False)
display(method_summary)


,method,resolved_edges_before_global_conflict_check
3,evidence2_unique,199691
0,points_unique,195844
1,date_unique,99011
2,group_unique,51592
4,user_singleton,7390


## 7. Финальная таблица соответствия

Финальная таблица строится через left join к полному списку уникальных `(user_id, course_id)` из `users_courses` после teacher filter. Поэтому если связь не восстановлена, строка не теряется, а `users_course_id` остается `NA`.


In [9]:
method_union = pd.concat([edges.assign(method=name) for name, edges in method_edges.items()], ignore_index=True).drop_duplicates(subset=["user_id", "course_id", "users_course_id", "method"])

edge_methods = method_union.groupby(["user_id", "course_id", "users_course_id"])["method"].agg(lambda s: "|".join(sorted(set(s)))).reset_index()

clean_mapping, conflict_user_course, conflict_user_ucid = remove_conflicts(edge_methods[["user_id", "course_id", "users_course_id"]])
clean_mapping = clean_mapping.merge(edge_methods, on=["user_id", "course_id", "users_course_id"], how="left")

result_df = uc_pairs[["user_id", "course_id"]].merge(clean_mapping, on=["user_id", "course_id"], how="left").sort_values(["user_id", "course_id"]).reset_index(drop=True)
result_df["users_course_id"] = pd.to_numeric(result_df["users_course_id"], errors="coerce").astype("Int64")
result_df["mapping_status"] = np.where(result_df["users_course_id"].notna(), "resolved", "unresolved")
result_df = result_df.rename(columns={"method": "resolution_method"})

mapping_summary = pd.DataFrame([
    {"metric": "raw_users_courses_pairs_before_teacher_filter", "value": raw_users_courses_pairs},
    {"metric": "student_users_courses_pairs_after_teacher_filter", "value": len(uc_pairs)},
    {"metric": "resolved_pairs", "value": int(result_df["users_course_id"].notna().sum())},
    {"metric": "unresolved_pairs", "value": int(result_df["users_course_id"].isna().sum())},
    {"metric": "resolved_share", "value": float(result_df["users_course_id"].notna().mean())},
    {"metric": "conflict_user_course_pairs_removed", "value": len(conflict_user_course)},
    {"metric": "conflict_user_ucid_pairs_removed", "value": len(conflict_user_ucid)},
])

print("Mapping summary:")
display(mapping_summary)
print("Global conflicts removed from final mapping:")
display(conflict_user_course)
display(conflict_user_ucid)
print("Result sample:")
display(result_df.head(20))


Mapping summary:


,metric,value
0,raw_users_courses_pairs_before_teacher_filter,290835.00000
1,student_users_courses_pairs_after_teacher_filter,267206.00000
2,resolved_pairs,207977.00000
3,unresolved_pairs,59229.00000
4,resolved_share,0.77834
5,conflict_user_course_pairs_removed,0.00000
6,conflict_user_ucid_pairs_removed,2.00000


Global conflicts removed from final mapping:


,user_id,course_id,users_course_id_count


,user_id,users_course_id,course_id_count
3393,671864,463215,2
7038,678685,476135,2


Result sample:


,user_id,course_id,users_course_id,resolution_method,mapping_status
0,665740,754,<NA>,NaN,unresolved
1,665740,170000688,<NA>,NaN,unresolved
2,665741,754,<NA>,NaN,unresolved
3,665749,50000592,<NA>,NaN,unresolved
4,665750,754,<NA>,NaN,unresolved
5,665753,754,<NA>,NaN,unresolved
6,665753,50000592,<NA>,NaN,unresolved
7,665755,170000688,<NA>,NaN,unresolved
8,665757,170000688,<NA>,NaN,unresolved
9,665758,50000592,<NA>,NaN,unresolved


## 8. Метрики покрытия bridge для целевых таблиц

Здесь отвечаем на практический вопрос: насколько теперь можно ходить из `users_courses` к таблицам с `users_course_id`. Для `user_access_histories` показаны две зоны: `total` для всей таблицы и `observed_student_scope` только для `users_course_id`, которые встречались у учеников в `user_lessons` или `wk_users_courses_actions` после teacher filter.


In [10]:
resolved_ids = set(result_df["users_course_id"].dropna().astype("int64"))
observed_student_ucids = set(all_user_ucid["users_course_id"].dropna().astype("int64"))

# Schitaem pokrytie po total i po observed student scope.
def bridge_coverage(table_name: str, df: pd.DataFrame) -> dict:
    ids_total = set(df["users_course_id"].dropna().astype("int64"))
    rows_total = len(df)
    covered_rows_total = int(df["users_course_id"].isin(resolved_ids).sum())
    observed_scope_ids = ids_total & observed_student_ucids
    observed_scope_rows = int(df["users_course_id"].isin(observed_student_ucids).sum())
    observed_scope_covered_rows = int(df["users_course_id"].isin(observed_student_ucids & resolved_ids).sum())
    return {
        "table": table_name,
        "unique_ids_total": len(ids_total),
        "covered_unique_ids_total": len(ids_total & resolved_ids),
        "covered_unique_share_total": len(ids_total & resolved_ids) / len(ids_total) if ids_total else 0.0,
        "rows_total": rows_total,
        "covered_rows_total": covered_rows_total,
        "covered_row_share_total": covered_rows_total / rows_total if rows_total else 0.0,
        "observed_scope_unique_ids": len(observed_scope_ids),
        "observed_scope_covered_unique_ids": len(observed_scope_ids & resolved_ids),
        "observed_scope_covered_unique_share": len(observed_scope_ids & resolved_ids) / len(observed_scope_ids) if observed_scope_ids else 0.0,
        "observed_scope_rows": observed_scope_rows,
        "observed_scope_covered_rows": observed_scope_covered_rows,
        "observed_scope_covered_row_share": observed_scope_covered_rows / observed_scope_rows if observed_scope_rows else 0.0,
    }

coverage_df = pd.DataFrame([
    bridge_coverage("user_lessons", dfs["user_lessons"]),
    bridge_coverage("wk_users_courses_actions", dfs["wk_users_courses_actions"]),
    bridge_coverage("user_access_histories", dfs["user_access_histories"]),
])

display(coverage_df)

observed_scope_summary = pd.DataFrame([
    {"metric": "observed_student_users_course_id", "value": len(observed_student_ucids)},
    {"metric": "resolved_observed_student_users_course_id", "value": len(observed_student_ucids & resolved_ids)},
    {"metric": "resolved_observed_student_users_course_id_share", "value": len(observed_student_ucids & resolved_ids) / len(observed_student_ucids) if observed_student_ucids else 0.0},
])
display(observed_scope_summary)


,table,unique_ids_total,covered_unique_ids_total,covered_unique_share_total,rows_total,covered_rows_total,covered_row_share_total,observed_scope_unique_ids,observed_scope_covered_unique_ids,observed_scope_covered_unique_share,observed_scope_rows,observed_scope_covered_rows,observed_scope_covered_row_share
0,user_lessons,208672,207971,0.996641,3008595,3002233,0.997885,208672,207971,0.996641,3008595,3002233,0.997885
1,wk_users_courses_actions,208565,207883,0.996730,12633684,12601087,0.997420,208565,207883,0.996730,12633684,12601087,0.997420
2,user_access_histories,290784,207976,0.715225,667124,568901,0.852767,208646,207976,0.996789,569944,568901,0.998170


,metric,value
0,observed_student_users_course_id,208685.000000
1,resolved_observed_student_users_course_id,207977.000000
2,resolved_observed_student_users_course_id_share,0.996607


## 9. Диагностика нерешенных пар

Нерешенные пары делятся на три основные группы: у пользователя нет наблюдаемого `users_course_id`; для конкретной пары нет надежного сигнала; кандидаты были, но связь осталась неоднозначной или конфликтной.


In [11]:
candidate_edge_count = edge_evidence.groupby(["user_id", "course_id"]).size().rename("candidate_edge_count").reset_index()

unresolved_diag = (
    result_df.loc[result_df["users_course_id"].isna(), ["user_id", "course_id"]]
    .merge(user_course_count, on="user_id", how="left")
    .merge(user_ucid_count, on="user_id", how="left")
    .merge(candidate_edge_count, on=["user_id", "course_id"], how="left")
)

unresolved_diag["candidate_edge_count"] = unresolved_diag["candidate_edge_count"].fillna(0).astype(int)
unresolved_diag["reason_bucket"] = np.select(
    [unresolved_diag["observed_users_course_id_count"].isna(), unresolved_diag["candidate_edge_count"].eq(0), unresolved_diag["candidate_edge_count"].gt(0)],
    ["no_observed_users_course_id_for_user", "no_signal_candidate_for_pair", "ambiguous_or_conflict_candidate"],
    default="other",
)

unresolved_reason_summary = unresolved_diag["reason_bucket"].value_counts().rename_axis("reason_bucket").reset_index(name="pair_count")
unresolved_course_summary = unresolved_diag.groupby("course_id").size().rename("unresolved_pair_count").reset_index().sort_values("unresolved_pair_count", ascending=False).reset_index(drop=True)

display(unresolved_reason_summary)
display(unresolved_course_summary.head(30))


,reason_bucket,pair_count
0,ambiguous_or_conflict_candidate,28775
1,no_observed_users_course_id_for_user,23340
2,no_signal_candidate_for_pair,7114


,course_id,unresolved_pair_count
0,763,27631
1,836,9708
2,770,6225
3,50000592,4470
4,170000688,2642
5,1061,2069
6,1096,1908
7,771,918
8,1033,633
9,1062,409


## 10. Sohranenie rezultatov

Sohranyaem tolko v `data/AGENT`, ne menyaya `data/raw` i polzovatelskie notebook-i.


In [ ]:
minimal_mapping_path = AGENT_DIR / "resolve_user_course_mapping.csv"
detailed_mapping_path = AGENT_DIR / "resolve_user_course_mapping_detailed.csv"
metrics_path = AGENT_DIR / "resolve_user_course_metrics.csv"
coverage_path = AGENT_DIR / "resolve_user_course_table_coverage.csv"
unresolved_reasons_path = AGENT_DIR / "resolve_user_course_unresolved_reasons.csv"
unresolved_courses_path = AGENT_DIR / "resolve_user_course_unresolved_courses.csv"
conflict_user_course_path = AGENT_DIR / "resolve_user_course_conflicts_user_course.csv"
conflict_user_ucid_path = AGENT_DIR / "resolve_user_course_conflicts_user_ucid.csv"

result_df[["user_id", "course_id", "users_course_id"]].to_csv(minimal_mapping_path, index=False, encoding="utf-8", na_rep="NA")
result_df.to_csv(detailed_mapping_path, index=False, encoding="utf-8", na_rep="NA")
mapping_summary.to_csv(metrics_path, index=False, encoding="utf-8")
coverage_df.to_csv(coverage_path, index=False, encoding="utf-8")
unresolved_reason_summary.to_csv(unresolved_reasons_path, index=False, encoding="utf-8")
unresolved_course_summary.to_csv(unresolved_courses_path, index=False, encoding="utf-8")
conflict_user_course.to_csv(conflict_user_course_path, index=False, encoding="utf-8")
conflict_user_ucid.to_csv(conflict_user_ucid_path, index=False, encoding="utf-8")

saved_files = pd.DataFrame([
    {"artifact": "minimal_mapping", "path": str(minimal_mapping_path)},
    {"artifact": "detailed_mapping", "path": str(detailed_mapping_path)},
    {"artifact": "mapping_metrics", "path": str(metrics_path)},
    {"artifact": "table_coverage", "path": str(coverage_path)},
    {"artifact": "unresolved_reasons", "path": str(unresolved_reasons_path)},
    {"artifact": "unresolved_courses", "path": str(unresolved_courses_path)},
    {"artifact": "conflicts_user_course", "path": str(conflict_user_course_path)},
    {"artifact": "conflicts_user_ucid", "path": str(conflict_user_ucid_path)},
])
display(saved_files)


## Итоговый вывод

В текущих данных после удаления teacher/account-agent восстановление получилось частичным, но практически полностью покрывает activity-таблицы:

* финальная таблица содержит все уникальные пары `(user_id, course_id)` из `users_courses` после teacher filter;
* для нерешенных пар `users_course_id` оставлен как `NA`;
* конфликтные кандидаты не выбирались вручную и были удалены из mapping-а;
* для `user_lessons` и `wk_users_courses_actions` bridge можно использовать почти для всех наблюдаемых `users_course_id`;
* для `user_access_histories` глобальное покрытие ниже, потому что таблица не содержит `user_id` и включает `users_course_id`, которые не наблюдались в очищенных activity-таблицах; внутри observed student scope покрытие почти полное.

Практическая рекомендация: для дальнейших merge использовать `data/AGENT/resolve_user_course_mapping.csv` как bridge и сохранять `NA`-строки как нерешенные, а не заполнять их эвристически.


## Контрольные числа текущего запуска

Кодовые ячейки были проверочно выполнены в Python через `nbformat`/`exec`, так как в окружении нет `jupyter`/`nbconvert` CLI.

Результат на текущих `data/raw` после teacher filter:

* всего уникальных student-пар `(user_id, course_id)` из `users_courses`: 267206;
* однозначно восстановлено: 207977 пар;
* осталось `NA`: 59229 пар;
* доля восстановления по всем student-парам: 0.77834;
* глобальные конфликты удалены: 0 конфликтов по `(user_id, course_id)` и 2 конфликта по `(user_id, users_course_id)`;
* покрытие `user_lessons`: 207971 из 208672 unique `users_course_id`, 3002233 из 3008595 строк;
* покрытие `wk_users_courses_actions`: 207883 из 208565 unique `users_course_id`, 12601087 из 12633684 строк;
* покрытие `user_access_histories` total: 207976 из 290784 unique `users_course_id`, 568901 из 667124 строк;
* покрытие `user_access_histories` внутри observed student scope: 207976 из 208646 unique `users_course_id`, 568901 из 569944 строк.
